# Smart Energy Model — demonstracja kodu (obrona)

**Autor:** Marta Gałuszka · **2026-08**

Jeden liniowy notebook do **Run All** na obronie: widać kod od danych → cech → model → wynik → **dashboard / app**.

| Gdzie więcej | Plik |
|--------------|------|
| Pełny research ML | [`02_ML_predykcja_PV.ipynb`](02_ML_predykcja_PV.ipynb) |
| Slajdy narracyjne (MLOps, live tydzień) | [`03_prezentacja_dyplomowa.ipynb`](03_prezentacja_dyplomowa.ipynb) |
| Raport Markdown z CSV (bez treningu) | [`05_raport_wynikow.ipynb`](05_raport_wynikow.ipynb) |
| Panel Streamlit | [`dashboard/app.py`](../dashboard/app.py) |
| Dokumentacja | [`docs/02_ML_predykcja_PV.md`](../docs/02_ML_predykcja_PV.md) |


In [ ]:
# 0) Setup — uruchom najpierw (lub Run All)
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

plt.rcParams['figure.figsize'] = (10, 4)
%matplotlib inline

def _find_root() -> Path:
    here = Path.cwd().resolve()
    for cand in (here, here.parent):
        if (cand / 'src').is_dir() and (cand / 'docs').is_dir():
            return cand
    return here.parent if here.name == 'notebooks' else here

ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

DATA = ROOT / 'data' / 'processed'
FIGURES = ROOT / 'reports' / 'figures'
MODELS = ROOT / 'models'

def show_table(df):
    d = df.copy()
    for c in d.columns:
        if pd.api.types.is_float_dtype(d[c]):
            d[c] = d[c].map(lambda x: f'{x:.3f}' if pd.notna(x) else '')
    cols = [str(c) for c in d.columns]
    lines = ['| ' + ' | '.join(cols) + ' |', '| ' + ' | '.join(['---'] * len(cols)) + ' |']
    for _, row in d.iterrows():
        lines.append('| ' + ' | '.join(str(v) for v in row.tolist()) + ' |')
    display(Markdown('\n'.join(lines)))

print('ROOT:', ROOT)


ROOT: /path/to/smart-energy-model


## 1. Wczytanie danych treningowych

Ten sam loader co produkcja: **16 cech**, target = Δ`PVEnergyTotal` (PVE).


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(ROOT / '.env')

from sklearn.model_selection import train_test_split

from src.features.pv_features_hourly_extended import (
    HOURLY_FEATURE_COLUMNS_PRODUCTION,
    TARGET_COLUMN,
    load_hourly_training_frame_extended,
)

TRAIN_START, TRAIN_END = '2025-06-01', '2026-05-31'
lat = float(os.getenv('WEATHER_LAT', '50.06'))  # fallback: Kraków (dokładne GPS w .env)
lon = float(os.getenv('WEATHER_LON', '19.94'))

frame = load_hourly_training_frame_extended(latitude=lat, longitude=lon)
frame = frame[(frame['day'] >= TRAIN_START) & (frame['day'] <= TRAIN_END)].copy()

days = frame['day'].unique()
days_train, days_test = train_test_split(days, test_size=0.2, random_state=42, shuffle=True)
tr = frame['day'].isin(days_train)
te = frame['day'].isin(days_test)

feats = list(HOURLY_FEATURE_COLUMNS_PRODUCTION)
X_tr = frame.loc[tr, feats].replace([np.inf, -np.inf], np.nan)
y_tr = frame.loc[tr, TARGET_COLUMN]
X_te = frame.loc[te, feats].replace([np.inf, -np.inf], np.nan)
y_te = frame.loc[te, TARGET_COLUMN]

print(f'Godziny: {len(frame):,} | dni: {frame["day"].nunique()}')
print(f'Train: {tr.sum():,} h ({len(days_train)} dni) | Test: {te.sum():,} h ({len(days_test)} dni)')
print(f'Target: {TARGET_COLUMN}')
display(frame[['day', 'hour', TARGET_COLUMN, 'radiation_wm2', 'cloud_cover_pct']].head(8))


FoxESS-Cloud Open API version 2.9.15


/path/to/smart-energy-model/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


✓ Target godzinowy = PVEnergyTotal (Δ licznika, jak w app): 422 dni, 4719 godzin


✓ Dodano cechy geometrii paneli (tilt=35°, azymut=180°)


✓ Dodano flagi śniegu z modelu topnienia (dni ze śniegiem: 35 / 422)


✓ Dodano flagę mgły (dni z mgłą: 146 / 422)
📊 Statystyki godzin produkcji:
   Najwcześniejsza: 5:00
   Najpóźniejsza: 20:00
   Średni wschód słońca: 5.60
   Średni zachód słońca: 19.29
Godziny: 3,505 | dni: 356
Train: 2,779 h (284 dni) | Test: 726 h (72 dni)
Target: pv_kwh_hour


,day,hour,pv_kwh_hour,radiation_wm2,cloud_cover_pct
0,2025-06-01,5,0.2,2.0,100.0
1,2025-06-01,6,0.2,44.0,94.0
2,2025-06-01,7,0.2,139.0,95.0
3,2025-06-01,8,4.1,283.0,28.0
4,2025-06-01,9,4.5,479.0,10.0
5,2025-06-01,10,4.6,630.0,19.0
6,2025-06-01,11,4.9,745.0,17.0
7,2025-06-01,12,3.7,828.0,42.0


## 2. EDA — produkcja dzienna i korelacja z pogodą

Okres **2025-06-01 → dziś**: suma dzienna PV (`pv_kwh_solar`) vs suma radiacji / zachmurzenie z Open-Meteo (ICON).


In [ ]:
from src.data.household_context import FOXESS_RELIABLE_START
from src.data.weather_api import load_daily_pv, load_daily_weather

DB_PATH = str(ROOT / 'data' / 'energy_model.db')
WX_START = FOXESS_RELIABLE_START.isoformat()
WX_END = frame['day'].max()

weather = load_daily_weather(DB_PATH, WX_START, WX_END)
pv_daily = load_daily_pv(DB_PATH, WX_START, WX_END)
daily = weather.merge(pv_daily, on='day', how='inner')
PV_COL = 'pv_kwh_solar'

corr_rad = daily[PV_COL].corr(daily['radiation_kwh_m2'])
corr_cloud = daily[PV_COL].corr(daily['cloud_cover_avg'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(pd.to_datetime(daily['day']), daily[PV_COL], lw=0.8)
axes[0].set_title('Produkcja PV — suma dzienna (kWh)')
axes[0].set_xlabel('Dzień')
axes[0].grid(alpha=0.3)

axes[1].scatter(daily['radiation_kwh_m2'], daily[PV_COL], alpha=0.35, s=18)
axes[1].set_title('PV vs suma radiacji dzienna')
axes[1].set_xlabel('Radiacja [kWh/m²/d]')
axes[1].set_ylabel('PV (kWh/d)')
axes[1].text(0.05, 0.92, f'r = {corr_rad:.2f}', transform=axes[1].transAxes)
print(f'Okres {WX_START} → {WX_END} | n={len(daily)}')
print(f'PV ↔ radiacja: r={corr_rad:.3f} | PV ↔ zachmurzenie: r={corr_cloud:.3f}')
print('Korelacja radiacji > 0,7 → dobry sygnał pod RF (16 cech).')
plt.tight_layout()
plt.show()


Okres 2025-06-01 → 2026-05-31 | n=365
PV ↔ radiacja: r=0.925 | PV ↔ zachmurzenie: r=-0.686
Korelacja radiacji > 0,7 → dobry sygnał pod RF (16 cech).


## 3. Feature engineering — 16 cech produkcyjnych

Definicja w `src/features/pv_features_hourly_extended.py`.


In [ ]:
print(f'Liczba cech: {len(feats)}')
for i, c in enumerate(feats, 1):
    print(f'{i:2d}. {c}')


Liczba cech: 16
 1. hour
 2. temp_c
 3. humidity_pct
 4. cloud_cover_pct
 5. radiation_wm2
 6. wind_speed_ms
 7. sunrise_hour
 8. sunset_hour
 9. day_length_hours
10. hours_since_sunrise
11. hours_until_sunset
12. sun_position
13. is_daylight
14. snow_on_panels
15. snow_on_panels_prev
16. likely_fog_day


## 4. Porównanie algorytmów — `.fit()` Ridge / RF / XGBoost

Split **80/20 po dniach** — protokół `scripts/analysis/compare_algorithms_hourly.py`.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor

from src.models.pv_hourly_predictor import (
    RF_MAX_DEPTH, RF_MAX_FEATURES, RF_MIN_SAMPLES_LEAF,
    RF_MIN_SAMPLES_SPLIT, RF_N_ESTIMATORS, RF_RANDOM_STATE,
)

def gap_verdict(gap: float) -> str:
    if gap < 0.15:
        return 'OK — niski gap'
    if gap < 0.35:
        return 'Uwaga — lekki gap'
    return 'Odrzuć — duży gap'

def eval_pipe(name, pipe):
    pipe.fit(X_tr, y_tr)
    pred_tr = pipe.predict(X_tr)
    pred_te = pipe.predict(X_te)
    mae_tr = mean_absolute_error(y_tr, pred_tr)
    mae_te = mean_absolute_error(y_te, pred_te)
    return {
        'model': name,
        'test_mae_kwh_h': mae_te,
        'test_r2': r2_score(y_te, pred_te),
        'gap_kwh_h': mae_te - mae_tr,
        'werdykt': gap_verdict(mae_te - mae_tr),
    }, pred_te

models = {
    'Ridge': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', Ridge(alpha=100.0, random_state=42)),
    ]),
    'Random Forest': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', RandomForestRegressor(
            n_estimators=RF_N_ESTIMATORS,
            max_depth=RF_MAX_DEPTH,
            min_samples_leaf=RF_MIN_SAMPLES_LEAF,
            min_samples_split=RF_MIN_SAMPLES_SPLIT,
            max_features=RF_MAX_FEATURES,
            random_state=RF_RANDOM_STATE,
            n_jobs=-1,
        )),
    ]),
    'XGBoost': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', XGBRegressor(
            n_estimators=200, max_depth=6, learning_rate=0.1,
            subsample=0.9, colsample_bytree=0.9, random_state=42,
            objective='reg:squarederror', n_jobs=-1,
        )),
    ]),
}

rows, preds = [], {}
for name, pipe in models.items():
    print(f'fit: {name}…', end=' ')
    row, pred = eval_pipe(name, pipe)
    rows.append(row)
    preds[name] = pred
    print('OK')

cmp = pd.DataFrame(rows).sort_values('test_mae_kwh_h')
print('\nWyniki (holdout godzinowy):')
show_table(cmp.round(3))
print('\n→ Wybór produkcyjny: Random Forest.')
rf_pred_te = preds['Random Forest']


fit: Ridge… OK
fit: Random Forest… 

/path/to/smart-energy-model/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/path/to/smart-energy-model/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/path/to/smart-energy-model/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/path/to/smart-energy-model/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/path/to/smart-energy-model/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/path/to/smart-energy-model/venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ 

OK
fit: XGBoost… 

OK

Wyniki (holdout godzinowy):


| model | test_mae_kwh_h | test_r2 | gap_kwh_h | werdykt |
| --- | --- | --- | --- | --- |
| Random Forest | 0.602 | 0.675 | 0.096 | OK — niski gap |
| XGBoost | 0.614 | 0.654 | 0.470 | Odrzuć — duży gap |
| Ridge | 0.831 | 0.526 | -0.003 | OK — niski gap |


→ Wybór produkcyjny: Random Forest.


## 5. Ablacja cech (skrót)


In [ ]:
abl_path = DATA / 'ablation_results.csv'
if abl_path.exists():
    abl = pd.read_csv(abl_path)
    cols = [c for c in ['phase', 'n_features', 'test_mae', 'test_r2', 'gap'] if c in abl.columns]
    show_table(abl[cols].tail(8).round(3))
else:
    print('Brak', abl_path, '— uruchom: python scripts/analysis/ablation_study.py')


|  |
|  |
|  |
|  |
|  |
|  |
|  |
|  |

## 6. Scatter — rzeczywistość vs prognoza (RF, holdout)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_te, rf_pred_te, alpha=0.25, s=12, c='#27ae60')
lim = max(y_te.max(), rf_pred_te.max()) * 1.05
ax.plot([0, lim], [0, lim], 'r--', lw=1, label='y = x')
ax.set_xlabel('Rzeczywistość (kWh/h)')
ax.set_ylabel('Prognoza RF (kWh/h)')
ax.set_title(f'Holdout — R² = {r2_score(y_te, rf_pred_te):.3f}, MAE = {mean_absolute_error(y_te, rf_pred_te):.3f} kWh/h')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 7. Walidacja operacyjna (closeout vs FoxESS)


In [ ]:
val_path = ROOT / 'data' / 'processed' / 'forecasts' / 'forecast_validation.csv'
if not val_path.exists():
    print('Brak', val_path)
else:
    val = pd.read_csv(val_path)
    val['target_day'] = pd.to_datetime(val['target_day'])
    sub = val.dropna(subset=['actual_pv_report', 'predicted_daily_raw']).copy()
    sub = sub[sub['actual_pv_report'] > 0.5]
    sub['ape_daily_%'] = (sub['predicted_daily_raw'] - sub['actual_pv_report']).abs() / sub['actual_pv_report'] * 100
    sub['ape_midday_%'] = (sub['predicted_midday_raw'] - sub['actual_pv_report']).abs() / sub['actual_pv_report'] * 100
    if 'predicted_daily_cs4' in sub.columns:
        sub['ape_cs4_%'] = (sub['predicted_daily_cs4'] - sub['actual_pv_report']).abs() / sub['actual_pv_report'] * 100

    era = sub[sub['target_day'] >= '2026-07-27']
    all_ = sub[sub['target_day'] >= '2026-07-14']
    end_label = sub['target_day'].max().strftime('%d.%m')

    summary = pd.DataFrame([
        {
            'okres': f'Era dual 27.07–{end_label}',
            'n_dni': len(era),
            'MAPE_5:00_%': era['ape_daily_%'].mean(),
            'MAPE_12:00_%': era['ape_midday_%'].mean(),
            'MAPE_CS4_%': era['ape_cs4_%'].mean() if 'ape_cs4_%' in era.columns else None,
        },
        {
            'okres': f'Całość 14.07–{end_label}',
            'n_dni': len(all_),
            'MAPE_5:00_%': all_['ape_daily_%'].mean(),
            'MAPE_12:00_%': all_['ape_midday_%'].mean(),
            'MAPE_CS4_%': all_['ape_cs4_%'].mean() if 'ape_cs4_%' in all_.columns else None,
        },
    ])
    show_table(summary.round(1))

    tail_cols = ['target_day', 'actual_pv_report', 'predicted_daily_raw', 'predicted_midday_raw']
    if 'predicted_daily_cs4' in sub.columns:
        tail_cols.append('predicted_daily_cs4')
    tail = sub.sort_values('target_day').tail(10)[tail_cols].copy()
    tail['ape_5:00_%'] = sub.sort_values('target_day').tail(10)['ape_daily_%'].round(1).values
    tail['ape_12:00_%'] = sub.sort_values('target_day').tail(10)['ape_midday_%'].round(1).values
    if 'ape_cs4_%' in sub.columns:
        tail['ape_CS4_%'] = sub.sort_values('target_day').tail(10)['ape_cs4_%'].round(1).values
    tail['target_day'] = tail['target_day'].dt.strftime('%Y-%m-%d')
    print('\nOstatnie 10 dni:')
    show_table(tail.round(2))

    os_path = ROOT / 'data' / 'processed' / 'oneshot_rf_icon_vs_ukmo_daily.csv'
    if os_path.exists():
        os_df = pd.read_csv(os_path)
        os_today = os_df[os_df['day'] == '2026-08-07']
        if not os_today.empty:
            r = os_today.iloc[0]
            print('\nOneshot 07.08 (ICON vs UKMO, ten sam .joblib):')
            show_table(pd.DataFrame([
                {'model': 'ICON', 'prognoza_kWh': r['icon_seamless'], 'dokładność_%': r['dokl_icon']},
                {'model': 'UKMO', 'prognoza_kWh': r['ukmo_seamless'], 'dokładność_%': r['dokl_ukmo']},
                {'model': 'Actual app', 'prognoza_kWh': r['app_kwh'], 'dokładność_%': 100.0},
            ]).round(2))


| okres | n_dni | MAPE_5:00_% | MAPE_12:00_% | MAPE_CS4_% |
| --- | --- | --- | --- | --- |
| Era dual 27.07–10.08 | 15 | 9.4 | 9.2 | 10.5 |
| Całość 14.07–10.08 | 28 | 16.7 | 14.8 | 10.5 |


Ostatnie 10 dni:


| target_day | actual_pv_report | predicted_daily_raw | ape_pct |
| --- | --- | --- | --- |
| 2026-08-01 | 33.1 | 26.6 | 19.64 |
| 2026-08-02 | 16.4 | 14.16 | 13.66 |
| 2026-08-03 | 33.5 | 33.28 | 0.66 |
| 2026-08-04 | 34.6 | 33.9 | 2.02 |
| 2026-08-05 | 34.9 | 34.47 | 1.23 |
| 2026-08-06 | 33.2 | 34.12 | 2.77 |
| 2026-08-07 | 13.6 | 15.35 | 12.87 |
| 2026-08-08 | 25.4 | 30.89 | 21.61 |
| 2026-08-09 | 37.7 | 33.84 | 10.24 |
| 2026-08-10 | 28.9 | 30.57 | 5.78 |

## 8. Model produkcyjny — gdzie jest kod?

> **§4 (wyżej)** trenuje Ridge / RF / XGB na **demo split** (2025-06→2026-05) — to **porównanie algorytmów na obronie**.  
> **Produkcja** to **inna ścieżka**: GridSearch min-gap → zapis `.joblib` → prognoza w `mlops/forecast_pv.py`.

### Mapa plików — otwórz w IDE (Cursor: Cmd+P)

| Krok | Plik | Kluczowe symbole |
|------|------|------------------|
| **1. Cechy + macierz treningowa** | [`src/features/pv_features_hourly_extended.py`](../src/features/pv_features_hourly_extended.py) | `HOURLY_FEATURE_COLUMNS_PRODUCTION`, `load_hourly_training_frame_extended()` |
| **2. Pipeline + inferencja** | [`src/models/pv_hourly_predictor.py`](../src/models/pv_hourly_predictor.py) | `_default_pipeline()`, `PVHourlyPredictor.save/load/predict_days` |
| **3. Trening weekly (GridSearch)** | [`scripts/train/train_hourly_model_tuning.py`](../scripts/train/train_hourly_model_tuning.py) | sekcja `[6] Zapis modelu .joblib` |
| **4. Prognoza operacyjna** | [`mlops/forecast_pv.py`](../mlops/forecast_pv.py) | `predictor.load()` → `recommend_appliances()` |
| **5. Harmonogram** | [`mlops/daily_workflow.sh`](../mlops/daily_workflow.sh), [`mlops/train_dual_weekly.sh`](../mlops/train_dual_weekly.sh) | 5:00 / 12:00 / niedziela 04:30 |
| **6. API (app)** | [`api/services/forecast_ml.py`](../api/services/forecast_ml.py) | ten sam `.joblib` co MLOps |

### Artefakty na dysku

| Plik | Zawartość |
|------|-----------|
| `models/pv_hourly_model.joblib` | `{pipeline, feature_columns, latitude, longitude, report}` |
| `models/pv_hourly_model.metadata.json` | Test MAE, gap, hiperparametry, `train_end` |

### Odtworzenie w terminalu

```bash
cd smart-energy-model
./venv/bin/python scripts/train/train_hourly_model_tuning.py   # trening → .joblib
./venv/bin/python mlops/forecast_pv.py --days 3 --sync         # prognoza → pv_forecast.csv
```

Poniżej: **fragmenty kodu produkcyjnego** + załadowanie `.joblib` (wymaga Run All od §0).


In [ ]:
import json
from src.models.pv_hourly_predictor import (
    PVHourlyPredictor,
    _default_pipeline,
    RF_MAX_DEPTH,
    RF_MAX_FEATURES,
    RF_MIN_SAMPLES_LEAF,
    RF_MIN_SAMPLES_SPLIT,
    RF_N_ESTIMATORS,
    RF_RANDOM_STATE,
    metadata_path_for,
)

# --- Pipeline produkcyjny (src/models/pv_hourly_predictor.py::_default_pipeline) ---
prod_pipeline = _default_pipeline()
rf = prod_pipeline.named_steps['model']
print('Pipeline: SimpleImputer(median) → RandomForestRegressor')
print(f'  n_estimators={RF_N_ESTIMATORS}')
print(f'  max_depth={RF_MAX_DEPTH}')
print(f'  min_samples_leaf={RF_MIN_SAMPLES_LEAF}, min_samples_split={RF_MIN_SAMPLES_SPLIT}')
print(f'  max_features={RF_MAX_FEATURES}, random_state={RF_RANDOM_STATE}')

model_path = MODELS / 'pv_hourly_model.joblib'
if not model_path.exists():
    print('\n⚠️ Brak', model_path, '— weekly: ./mlops/train_dual_weekly.sh')
else:
    predictor = PVHourlyPredictor(model_path=str(model_path))
    predictor.load()
    pipe = predictor.pipeline

    print('\nZaładowano:', model_path.name)
    print('Kroki:', list(pipe.named_steps.keys()))
    print(f'Cechy ({len(predictor.feature_columns)}):', ', '.join(predictor.feature_columns[:4]), '…')

    meta_path = Path(metadata_path_for(str(model_path)))
    if meta_path.exists():
        meta = json.loads(meta_path.read_text(encoding='utf-8'))
        met = meta.get('metrics', {})
        print(f"\nMetadata · train_end {meta.get('train_end')} · saved {meta.get('saved_at', '')[:10]}")
        show_table(pd.DataFrame([{
            'Test MAE [kWh/h]': met.get('test_mae'),
            'Gap [kWh/h]': met.get('gap'),
            'Daily MAE [kWh/d]': met.get('daily_mae'),
            'Werdykt': met.get('verdict', '').replace('✅ ', ''),
        }]).round(3))

    # Prognoza na holdoucie z §1 (demo split — metryki .joblib = inne okno 80/20)
    X_prod = X_te[predictor.feature_columns].replace([np.inf, -np.inf], np.nan)
    prod_pred = np.clip(pipe.predict(X_prod), 0, None)
    print(f"\nInferencja na holdoucie (split demo §1): MAE={mean_absolute_error(y_te, prod_pred):.3f} kWh/h, "
          f"R²={r2_score(y_te, prod_pred):.3f}")


Załadowano: /path/to/smart-energy-model/models/pv_hourly_model.joblib
Typ: dict


In [ ]:
import inspect
from pathlib import Path

PROD_FILES = {
    'cechy (16)': ROOT / 'src/features/pv_features_hourly_extended.py',
    'pipeline + PVHourlyPredictor': ROOT / 'src/models/pv_hourly_predictor.py',
    'trening GridSearch': ROOT / 'scripts/train/train_hourly_model_tuning.py',
    'prognoza MLOps': ROOT / 'mlops/forecast_pv.py',
}

print('=== Pliki produkcyjne (istnieją w repo?) ===')
for label, path in PROD_FILES.items():
    print(f"  {'✓' if path.exists() else '✗'} {label}: {path.relative_to(ROOT)}")

print('\n--- _default_pipeline() — definicja RF produkcyjnego ---')
print(inspect.getsource(_default_pipeline).strip())

print('\n--- PVHourlyPredictor.save() — co trafia do .joblib ---')
print(inspect.getsource(PVHourlyPredictor.save).strip()[:900], '…')

print('\n--- forecast_pv.py (fragment main) — jak ładujemy model ---')
fv = (ROOT / 'mlops/forecast_pv.py').read_text(encoding='utf-8')
start = fv.find('predictor = PVHourlyPredictor')
print(fv[start:start + 520].strip())

## 9. Dashboard i aplikacja

Model z §8 trafia do **dwóch warstw UI**:

| Warstwa | Kod | Rola |
|---------|-----|------|
| **Panel Streamlit** | [`dashboard/app.py`](../dashboard/app.py) | Operacje: walidacja prognozy vs app, notatki pogodowe, faktury Tauron |
| **Aplikacja mobilna** | [`mobile/`](../mobile/) + [`api/`](../api/) | Użytkownik: prognoza PV, bilans dachu, symulator rachunków, sugestie baterii |

### Uruchomienie dashboardu (obrona / demo)

```bash
cd smart-energy-model
source venv/bin/activate
streamlit run dashboard/app.py
# → http://localhost:8501
```

*(Backend API: `uvicorn api.main:app --port 8000` · frontend Ionic: `mobile/` → port 8100.)*

### Zrzuty aplikacji mobilnej

Poniżej: animacja tour + ekrany z `docs/images/app/` (jak w README).


<img src="../docs/images/app/app-tour.gif" width="900" alt="Smart Energy — tour aplikacji"/>

| Home · sync FoxESS | Prognoza dziś (profil godzinowy) |
|:---:|:---:|
| <img src="../docs/images/app/s1-sync.png" width="420" alt="Home"/> | <img src="../docs/images/app/s2-prognoza-dzis.png" width="420" alt="Prognoza dziś"/> |

| Closeout / inny dzień | Symulator rachunków |
|:---:|:---:|
| <img src="../docs/images/app/s3-prognoza-jutro.png" width="420" alt="Prognoza jutro"/> | <img src="../docs/images/app/s4-sugestie.png" width="420" alt="Symulator"/> |

**Powiązanie z ML:** prognoza godzinowa w app pochodzi z `mlops/forecast_pv.py` → ten sam `pv_hourly_model.joblib` co w §8.

## 10. Podsumowanie

1. Prognoza godzinowa PV (kWh/h) — FoxESS + Open-Meteo ICON.  
2. Target PVE = ta sama skala co aplikacja FoxESS.  
3. 16 cech po ablacji; Random Forest w produkcji (`§8`).  
4. MLOps → **dashboard Streamlit** (`§9`) + **app mobilna**; MAPE live w closeoucie (`§7`).

Slajdy: [`03_prezentacja_dyplomowa.ipynb`](03_prezentacja_dyplomowa.ipynb) · metryki: [`docs/STATUS_ML_MLOPS.md`](../docs/STATUS_ML_MLOPS.md)
